# Example Lightcurve Format

The purpose of this repository is to document an example of the requested lightcurve format provided for events detected by the ``cb2rges`` pipeline.  This format may be modified; this example is provided as a starting point. 

It was agreed within the RGES-PIT that the easiest format to read and write would be a FITS binary table.  

This file should consist of a PrimaryHDU and single binary table extension, with the columns as described below.
The rest of this notebook provides a simulated example of the file specified. 

In [2]:
from astropy.table import Table, Column
from astropy.time import Time, TimeDelta
import numpy as np

The header of the FITS file should contain at least the following keywords:

* GAIN: Detector gain used to convert ADU to e-
* DIACENS: DIA centroid shift in pixels
* PSFCENS: PSF centroid shift in pixels

The tabular data in the binary table should have the following columns:

* datetime:  DateTime in JD of the mid-point of a given exposure
* x: x-pixel position in the image
* y: y-pixel position in the image
* background subtracted flux (BSF): Flux in ADU measured from the background-subtracted Image
* BSF err: Uncertainty on the BSF
* DIA flux: Flux in ADU measured from the Difference Image
* DIA err: Uncertainty on the DIA flux

In [9]:
# Example header keywords 
header = {
    'GAIN': 1.5,
    'DIACENS': 0.002,
    'PSFCENS': 0.0004
}

In [11]:
# Example timeseries dataset
ndata = 100
start_time = Time('2027-01-01T00:00:00', format='isot')
dt = TimeDelta(15.0/(60.0*24.0), format='jd')
times = start_time + dt * np.arange(0, ndata, 1)
jd = Time(times, format='isot', scale='utc').jd

x = np.random.rand(ndata)*1000.0
y = np.random.rand(ndata)*1000.0
zp = 25.0
bsf = zp - 2.5 * np.log10(np.random.normal(loc=20.0, scale=4.0, size=ndata))
bsf_err = 0.0001 * bsf 
dia_flux = bsf 
dia_flux_err = bsf_err

In [12]:
# Create Astropy Table 
results = Table([
    Column(name='datetime', data=jd, dtype=np.float64),
    Column(name='x', data=x, dtype=np.float64),
    Column(name='y', data=y, dtype=np.float64),
    Column(name='bsf', data=bsf, dtype=np.float64),
    Column(name='bsf_err', data=bsf_err, dtype=np.float64),
    Column(name='dia_flux', data=dia_flux, dtype=np.float64),
    Column(name='dia_flux_err', data=dia_flux_err, dtype=np.float64)
])

results.meta = header

results

datetime,x,y,bsf,bsf_err,dia_flux,dia_flux_err
float64,float64,float64,float64,float64,float64,float64
2461406.5,226.12909687481763,379.8378858806709,21.576188996814267,0.0021576188996814266,21.576188996814267,0.0021576188996814266
2461406.5104166665,504.68997672708514,378.4237537764389,21.763614643173828,0.002176361464317383,21.763614643173828,0.002176361464317383
2461406.5208333335,160.9999880102021,341.426624091299,21.713026071656518,0.002171302607165652,21.713026071656518,0.002171302607165652
2461406.53125,934.150619577937,537.673228663016,22.225169512384728,0.002222516951238473,22.225169512384728,0.002222516951238473
2461406.5416666665,236.57159582256048,939.5437891646524,21.696055184120013,0.0021696055184120015,21.696055184120013,0.0021696055184120015
2461406.5520833335,268.2197295675781,542.3102505551124,21.804080498762136,0.002180408049876214,21.804080498762136,0.002180408049876214
2461406.5625,84.1179874143294,587.5539254786721,21.97108610585919,0.002197108610585919,21.97108610585919,0.002197108610585919
2461406.5729166665,80.64275850745095,652.8547492203783,21.719752624560343,0.0021719752624560344,21.719752624560343,0.0021719752624560344
2461406.5833333335,69.3684003824424,267.61622520506614,21.892486538419533,0.002189248653841953,21.892486538419533,0.002189248653841953


In [13]:
# Save as a FITS file 
results.write('example_event_lightcurve.fits', overwrite=True)

In [14]:
# Loading a FITS event lightcurve 
t = Table.read('example_event_lightcurve.fits')
t

datetime,x,y,bsf,bsf_err,dia_flux,dia_flux_err
float64,float64,float64,float64,float64,float64,float64
2461406.5,226.12909687481763,379.8378858806709,21.576188996814267,0.0021576188996814266,21.576188996814267,0.0021576188996814266
2461406.5104166665,504.68997672708514,378.4237537764389,21.763614643173828,0.002176361464317383,21.763614643173828,0.002176361464317383
2461406.5208333335,160.9999880102021,341.426624091299,21.713026071656518,0.002171302607165652,21.713026071656518,0.002171302607165652
2461406.53125,934.150619577937,537.673228663016,22.225169512384728,0.002222516951238473,22.225169512384728,0.002222516951238473
2461406.5416666665,236.57159582256048,939.5437891646524,21.696055184120013,0.0021696055184120015,21.696055184120013,0.0021696055184120015
2461406.5520833335,268.2197295675781,542.3102505551124,21.804080498762136,0.002180408049876214,21.804080498762136,0.002180408049876214
2461406.5625,84.1179874143294,587.5539254786721,21.97108610585919,0.002197108610585919,21.97108610585919,0.002197108610585919
2461406.5729166665,80.64275850745095,652.8547492203783,21.719752624560343,0.0021719752624560344,21.719752624560343,0.0021719752624560344
2461406.5833333335,69.3684003824424,267.61622520506614,21.892486538419533,0.002189248653841953,21.892486538419533,0.002189248653841953


In [15]:
t.meta

{'GAIN': 1.5, 'DIACENS': 0.002, 'PSFCENS': 0.0004}